# Boston Marathon Results Scraper
Scrapes results from [boston.r.mikatiming.com](https://boston.r.mikatiming.com)

In [1]:
# Install dependencies if needed
# %pip install requests beautifulsoup4 pandas selenium

In [2]:
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## Config — set your years and options here

In [3]:
YEARS = [2024]  # list of years to scrape
EVENT = "R"  # R=Runners, W=Wheelchairs, H=Handcycles
OUTPUT_DIR = Path(".")  # where to save CSVs
DELAY = 1.0  # seconds between requests
FORCE_SELENIUM = False  # set True if requests approach returns empty results

## Scraper functions

In [4]:
BASE_URL = "https://boston.r.mikatiming.com/{year}/"
RESULTS_PER_PAGE = 100

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}


def fetch_page_html(year, event, num_results, page):
    params = {
        "pid": "list",
        "pidp": "start",
        "event": event,
        "num_results": num_results,
        "page": page,
    }
    resp = requests.get(
        BASE_URL.format(year=year), params=params, headers=HEADERS, timeout=30
    )
    resp.raise_for_status()
    return resp.text


def parse_table(html):
    """Return (rows: list[dict], total_count: int | None)."""
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    if not table:
        return [], None

    headers = [c.get_text(strip=True) for c in table.find("tr").find_all(["th", "td"])]

    rows = []
    for tr in table.find_all("tr")[1:]:
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if cells and len(cells) == len(headers):
            rows.append(dict(zip(headers, cells)))

    total = None
    pager = soup.find(string=re.compile(r"of\s+[\d,]+"))
    if pager:
        m = re.search(r"of\s+([\d,]+)", pager)
        if m:
            total = int(m.group(1).replace(",", ""))

    return rows, total


def scrape_with_requests(year, event, delay):
    all_rows = []
    page = 0
    while True:
        print(f"  [requests] page {page}...", end="", flush=True)
        html = fetch_page_html(year, event, RESULTS_PER_PAGE, page)
        rows, total = parse_table(html)

        if not rows:
            print(" empty — page may require JavaScript")
            return None  # signal to fall back to Selenium

        all_rows.extend(rows)
        print(f" {len(rows)} rows (total so far: {len(all_rows)})")

        if len(rows) < RESULTS_PER_PAGE:
            break
        if total and len(all_rows) >= total:
            break

        page += 1
        time.sleep(delay)

    return all_rows


def scrape_with_selenium(year, event, delay):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import Select, WebDriverWait

    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1280,900")

    driver = webdriver.Chrome(options=opts)
    wait = WebDriverWait(driver, 20)
    all_rows = []

    try:
        driver.get(BASE_URL.format(year=year))
        time.sleep(3)

        try:
            Select(driver.find_element(By.NAME, "num_results")).select_by_value("1000")
            time.sleep(2)
        except Exception:
            pass

        page = 0
        while True:
            print(f"  [selenium] page {page}...", end="", flush=True)
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows, total = parse_table(driver.page_source)

            if not rows:
                print(" no table found")
                break

            all_rows.extend(rows)
            print(f" {len(rows)} rows (total so far: {len(all_rows)})")

            if total and len(all_rows) >= total:
                break

            try:
                next_btn = driver.find_element(
                    By.CSS_SELECTOR, "a.next, a[aria-label='Next'], .pagination-next a"
                )
                if "disabled" in (next_btn.get_attribute("class") or ""):
                    break
                next_btn.click()
                time.sleep(delay + 1)
                page += 1
            except Exception:
                break
    finally:
        driver.quit()

    return all_rows


def scrape_year(year, event, output_dir, delay, force_selenium):
    print(f"\nScraping {year} (event={event})...")

    rows = None if force_selenium else scrape_with_requests(year, event, delay)

    if rows is None:
        print("  Falling back to Selenium...")
        rows = scrape_with_selenium(year, event, delay)

    if not rows:
        print(f"  No results found for {year}.")
        return None

    df = pd.DataFrame(rows)
    out_path = Path(output_dir) / f"results_{year}.csv"
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df):,} rows -> {out_path}")
    return df

## Run the scraper

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results = {}
for year in YEARS:
    df = scrape_year(year, EVENT, OUTPUT_DIR, DELAY, FORCE_SELENIUM)
    if df is not None:
        results[year] = df


Scraping 2024 (event=R)...
  [requests] page 0... empty — page may require JavaScript
  Falling back to Selenium...
  [selenium] page 0...

TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x0000000100ee5ebc cxxbridge1$str$ptr + 3213376
1   chromedriver                        0x0000000100edde8c cxxbridge1$str$ptr + 3180560
2   chromedriver                        0x00000001009a38d4 _RNvCs10ygTOo3JCa_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75096
3   chromedriver                        0x00000001009eb8b4 _RNvCs10ygTOo3JCa_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 369976
4   chromedriver                        0x0000000100a2b84c _RNvCs10ygTOo3JCa_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 632016
5   chromedriver                        0x00000001009e127c _RNvCs10ygTOo3JCa_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 327424
6   chromedriver                        0x0000000100ea3f60 cxxbridge1$str$ptr + 2943204
7   chromedriver                        0x0000000100ea7744 cxxbridge1$str$ptr + 2957512
8   chromedriver                        0x0000000100e88e50 cxxbridge1$str

## Preview results

In [ ]:
for year, df in results.items():
    print(f"\n--- {year} ({len(df):,} rows) ---")
    display(df.head())